In [3]:
# import os
#
# import dotenv
# from langchain_openai import  ChatOpenAI
#
#
# dotenv.load_dotenv()
# # os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_API_KEY")
# # os.environ["OPENAI_BASE_URL"] = os.getenv("OPEN_ROUTER_API_URL")
# os.environ["OPENAI_API_KEY"] = os.getenv("DEEP_SEEK_API_KEY")
# os.environ["OPENAI_BASE_URL"] = os.getenv("DEEP_SEEK_API_URL")
#
#
# llm = ChatOpenAI(model='deepseek-chat')
#
#
# response = llm.invoke("什么是大模型")
# print(response)
#
#


In [5]:
import os

import dotenv
from langchain_openai import  ChatOpenAI


dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPEN_ROUTER_API_URL")
# os.environ["OPENAI_API_KEY"] = os.getenv("DEEP_SEEK_API_KEY")
# os.environ["OPENAI_BASE_URL"] = os.getenv("DEEP_SEEK_API_URL")
llm = ChatOpenAI(model='deepseek-chat')

# 2. 使用提示词模板

In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system","你是一个世界级的技术文档编写者"),
    ("user","{input}")
])

chain = prompt | llm

message = chain.invoke({"input":"大模型中的LangChain是什么"})
print(message)

BadRequestError: Error code: 400 - {'error': {'message': 'deepseek-chat is not a valid model ID', 'code': 400}, 'user_id': 'user_30AB2TOun9D3cdX3NvOkqdx8iPX'}

# 3 使用输出解析器

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import  StrOutputParser, JsonOutputParser

llm  = ChatOpenAI(model='gpt-4o-mini')


prompt = ChatPromptTemplate.from_messages([
    ("system","你是一个世界级的技术文档编写者"),
    ("user","{input}")
])

output_parser = JsonOutputParser()
chain = prompt | llm | output_parser


chain.invoke({"input":"LangChain 是什么? 用JSON格式回复，问题用question，答案用answer"})


{'question': 'LangChain 是什么?',
 'answer': 'LangChain 是一个用于构建语言模型应用程序的框架。它提供了一系列工具和组件，使开发者能够轻松地集成自然语言处理（NLP）模型，处理和生成文本，创建对话式AI，优化工作流程，连接不同的数据源，以及构建复杂的应用程序。LangChain 的设计旨在简化与语言模型的交互，并促进多种应用的开发，例如聊天机器人、内容生成和数据分析。'}

In [8]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 使用分割器分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = [ """新华网上海10月24日电（记者 吴振东）复旦大学管理学院恢复建院四十周年院庆之际，全球管理教育院长论坛暨全球商科教育博览会24日开幕。来自美国麻省理工学院斯隆管理学院、耶鲁大学管理学院、伦敦商学院等近50所顶尖商学院及商科教育国际组织负责人、代表齐聚上海，共同回顾管理教育国际化成就、凝聚改革共识、探讨合作方向。"""]
print(len(documents))
split_docs = text_splitter.create_documents(documents)

# 向量存储 embeddings 会将 documents 中的每个文本片段转换为向量，并将这些向量存储在 FAISS
# 向量数据库中
vector = FAISS.from_documents(split_docs, embeddings)

1


In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

embeddings = OpenAIEmbeddings(
    model="qwen/qwen3-embedding-8b",  # 或其他 OpenRouter 支持的模型
)
# 使用分割器分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

documents = ["""新华网上海10月24日电(记者 吴振东)复旦大学管理学院恢复建院四十周年院庆之际,全球管理教育院长论坛暨全球商科教育博览会24日开幕。来自美国麻省理工学院斯隆管理学院、耶鲁大学管理学院、伦敦商学院等近50所顶尖商学院及商科教育国际组织负责人、代表齐聚上海,共同回顾管理教育国际化成就、凝聚改革共识、探讨合作方向。"""]

# 先分割文档
split_docs = text_splitter.create_documents(documents)
print(f"分割后的文档数量: {len(split_docs)}")

# 创建向量存储
vector = FAISS.from_documents(split_docs, embeddings)
print(vector)

分割后的文档数量: 1


In [10]:
from langchain_core.prompts import PromptTemplate
retriever = vector.as_retriever()
retriever.search_kwargs = {"k": 3}
docs = retriever.invoke("建设用地使用权是什么？")
# for i,doc in enumerate(docs):
# print(f"⭐第{i+1}条规定：")
# print(doc)
# 6.定义提示词模版
prompt_template = """
你是一个问答机器人。
你的任务是根据下述给定的已知信息回答用户问题。
确保你的回复完全依据下述已知信息。不要编造答案。
如果下述已知信息不足以回答用户的问题，请直接回复"我无法回答您的问题"。
已知信息:
{info}
用户问：
{question}
请用中文回答用户问题。
"""
# 7.得到提示词模版对象
template = PromptTemplate.from_template(prompt_template)
# 8.得到提示词对象
prompt = template.format(info=docs, question='复旦大学管理学院恢复多久了？')
# print(prompt)
## 9. 调用LLM
response = llm.invoke(prompt)
print(response.content)


复旦大学管理学院恢复建院四十周年。


# 5.6 使用Agent


In [19]:
from langchain_core.tools import create_retriever_tool


retriever_tool = create_retriever_tool(
retriever,
"CivilCodeRetriever",
"搜索有关中华人民共和国民法典的信息。关于中华人民共和国民法典的任何问题，您必须使用此工具!",
)
tools = [retriever_tool]

from langgraph.prebuilt import create_react_agent

# 使用 create_react_agent 替代已弃用的 create_openai_functions_agent
agent_executor = create_react_agent(llm, tools)

# 运行代理
result = agent_executor.invoke({"messages": [("human", "建设用地使用权是什么")]})
print(result["messages"][-1].content)




> Entering new AgentExecutor chain...


BadRequestError: Error code: 400 - {'error': {'message': '"functions" and "function_call" are deprecated in favor of "tools" and "tool_choice." To learn how to use tools, visit: https://openrouter.ai/docs/guides/features/tool-calling', 'code': 400, 'metadata': {'provider_name': 'Azure'}}, 'user_id': 'user_30AB2TOun9D3cdX3NvOkqdx8iPX'}